In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import KNNImputer

In [2]:
from google.colab import files
uploaded = files.upload()  # upload the 4 processed.*.data files again

Saving processed.cleveland.data to processed.cleveland.data
Saving processed.hungarian.data to processed.hungarian.data
Saving processed.switzerland.data to processed.switzerland.data
Saving processed.va.data to processed.va.data


In [3]:
columns = ["age","sex","cp","trestbps","chol","fbs","restecg",
           "thalach","exang","oldpeak","slope","ca","thal","target"]

cleveland = pd.read_csv("processed.cleveland.data", names=columns, na_values="?")
hungarian = pd.read_csv("processed.hungarian.data", names=columns, na_values="?")
switzerland = pd.read_csv("processed.switzerland.data", names=columns, na_values="?")
va = pd.read_csv("processed.va.data", names=columns, na_values="?")

cleveland["source"] = "cleveland"
hungarian["source"] = "hungarian"
switzerland["source"] = "switzerland"
va["source"] = "va"

combined = pd.concat([cleveland, hungarian, switzerland, va], ignore_index=True)
print(combined.shape)  # expect (920, 15)

(920, 15)


In [4]:
combined["chol"] = combined["chol"].replace(0, np.nan)
combined["trestbps"] = combined["trestbps"].replace(0, np.nan)
print(combined.isnull().sum())

age           0
sex           0
cp            0
trestbps     60
chol        202
fbs          90
restecg       2
thalach      55
exang        55
oldpeak      62
slope       309
ca          611
thal        486
target        0
source        0
dtype: int64


In [5]:
combined["ca_missing"] = combined["ca"].isnull().astype(int)
combined["thal_missing"] = combined["thal"].isnull().astype(int)
combined["slope_missing"] = combined["slope"].isnull().astype(int)
combined["chol_missing"] = combined["chol"].isnull().astype(int)
print(combined.shape)  # expect (920, 19)

(920, 19)


In [6]:
# Binary version
binary_full = combined.copy()
binary_full["target_binary"] = (binary_full["target"] > 0).astype(int)

X_bin = binary_full.drop(columns=["target", "target_binary", "source"])
y_bin = binary_full["target_binary"]

X_train_bin, X_test_bin, y_train_bin, y_test_bin = train_test_split(
    X_bin, y_bin, test_size=0.2, random_state=42, stratify=y_bin
)

# Multi-class version (only cleveland/switzerland/va, same as before)
multi_full = combined[combined["source"].isin(["cleveland","switzerland","va"])].copy()
X_multi = multi_full.drop(columns=["target", "source"])
y_multi = multi_full["target"]

X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    X_multi, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

print(X_train_bin.shape, X_test_bin.shape)
print(X_train_multi.shape, X_test_multi.shape)

(736, 17) (184, 17)
(500, 17) (126, 17)


In [7]:
# --- Binary dataset ---
median_cols = ["thalach", "trestbps"]
mode_cols = ["fbs", "exang", "restecg", "slope"]
knn_cols = ["age","sex","cp","trestbps","chol","fbs","restecg",
            "thalach","exang","oldpeak","slope","ca","thal"]

def fit_and_apply_imputation(X_train, X_test):
    X_train = X_train.copy()
    X_test = X_test.copy()

    # Step 1: median for small-gap numeric columns — learned from TRAIN only
    medians = X_train[median_cols].median()
    X_train[median_cols] = X_train[median_cols].fillna(medians)
    X_test[median_cols] = X_test[median_cols].fillna(medians)

    # Step 2: mode for small-gap categorical + slope — learned from TRAIN only
    modes = X_train[mode_cols].mode().iloc[0]
    X_train[mode_cols] = X_train[mode_cols].fillna(modes)
    X_test[mode_cols] = X_test[mode_cols].fillna(modes)

    # Step 3: chol per-hospital median — learned from TRAIN only
    # NOTE: 'source' column was dropped earlier for modeling, so we handle chol
    # using the global TRAIN median here (simpler, avoids needing 'source' at test time)
    chol_median = X_train["chol"].median()
    X_train["chol"] = X_train["chol"].fillna(chol_median)
    X_test["chol"] = X_test["chol"].fillna(chol_median)

    # Step 4: KNN imputer for ca/thal — fit on TRAIN only, applied to both
    imputer = KNNImputer(n_neighbors=5)
    X_train[knn_cols] = imputer.fit_transform(X_train[knn_cols])
    X_test[knn_cols] = imputer.transform(X_test[knn_cols])

    return X_train, X_test

X_train_bin, X_test_bin = fit_and_apply_imputation(X_train_bin, X_test_bin)
X_train_multi, X_test_multi = fit_and_apply_imputation(X_train_multi, X_test_multi)

print(X_train_bin.isnull().sum().sum(), X_test_bin.isnull().sum().sum())
print(X_train_multi.isnull().sum().sum(), X_test_multi.isnull().sum().sum())

0 0
0 0


In [10]:
X_train_bin.to_csv("X_train_binary.csv", index=False)
X_test_bin.to_csv("X_test_binary.csv", index=False)
y_train_bin.to_csv("y_train_binary.csv", index=False)
y_test_bin.to_csv("y_test_binary.csv", index=False)

X_train_multi.to_csv("X_train_multiclass.csv", index=False)
X_test_multi.to_csv("X_test_multiclass.csv", index=False)
y_train_multi.to_csv("y_train_multiclass.csv", index=False)
y_test_multi.to_csv("y_test_multiclass.csv", index=False)

from google.colab import files
for f in ["X_train_binary.csv","X_test_binary.csv","y_train_binary.csv","y_test_binary.csv",
          "X_train_multiclass.csv","X_test_multiclass.csv","y_train_multiclass.csv","y_test_multiclass.csv"]:
    files.download(f)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>